# PatchCore via anomalib — Patch-Level Anomaly Detection
## MGCLS Radio Source Anomaly Detection · University of the Western Cape · 2026

This notebook implements **PatchCore** using the
[anomalib](https://github.com/openvinotoolkit/anomalib) library
(Akcay et al., arXiv:2202.08341).

### Why anomalib over our custom implementation

| | Custom PatchCore (previous notebook) | anomalib PatchCore (this notebook) |
|---|---|---|
| Feature extraction | Global average pool of layer2 + layer3 | **Patch-level** spatial feature maps — preserves local structure |
| Aggregation | One 768-dim vector per image | Many patch vectors per image → richer memory bank |
| Backbone | WideResNet50 only | ResNet18, WideResNet50, EfficientNet — selectable |
| Memory bank | All vectors kept | **Coreset subsampling** (greedy furthest-point) built-in |
| Scoring | Mean k-NN distance | k-NN with re-weighting as in the original paper |

Patch-level aggregation is the key improvement. Instead of summarising an entire
radio image as one vector, anomalib keeps a feature vector for each spatial patch
(e.g. 28×28 = 784 patches for a 224×224 image). The anomaly score of an image is
the maximum patch-level distance to the memory bank — so a single unusual region
anywhere in the image gets flagged, even if the rest looks normal.

### Goal
Beat the Protege baseline (ROC-AUC = 0.8674) using only the raw images
and no human labels.


## 1. Install Dependencies

In [ ]:
import sys

# ── Install strategy ──────────────────────────────────────────────────────────
# anomalib's full install pulls in `av` (video library) which requires
# FFmpeg system libraries not available on most Macs.
# We only need: timm (backbone registry) + torch + torchvision.
# We replicate the one anomalib function we use (TimmFeatureExtractor)
# directly in this notebook — no anomalib install needed at all.

!{sys.executable} -m pip install torch torchvision --quiet
!{sys.executable} -m pip install timm --quiet          # backbone registry
!{sys.executable} -m pip install Pillow --quiet
print("Done.")


## 2. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as T
import timm                              # backbone registry (replaces anomalib)

from sklearn.preprocessing import normalize as sk_normalize, StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import pairwise_distances_chunked
from torch.utils.data import Dataset, DataLoader

from utils import load_features, load_catalogue, compute_metrics, BASE_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
print(f'timm   : {timm.__version__}')


## 3. Evaluation Helpers

In [ ]:
def compute_ind_sum(found_inds, all_inds):
    out = np.zeros(len(all_inds))
    for i in sorted(found_inds):
        out[i:] += 1
    return out


def cumulative_sum(anomaly_scores, labels):
    """Running count of true anomalies as we walk down the ranked list."""
    sorted_inds = anomaly_scores.sort_values(ascending=False).index
    anom_inds   = labels[labels == 1].index
    found_inds  = [np.where(sorted_inds == i)[0][0]
                   for i in anom_inds if i in sorted_inds]
    return compute_ind_sum(found_inds, sorted_inds)


def topk_recall(y_true, scores, k=100):
    ranked = scores.sort_values(ascending=False).index[:k]
    return y_true.loc[ranked].sum() / y_true.sum()


def show_results(methods_dict, y_true, labels_raw):
    """Standard evaluation table sorted by ROC-AUC."""
    rows = []
    for name, scores in methods_dict.items():
        idx = scores.index.intersection(y_true.index)
        m   = compute_metrics(y_true.loc[idx], scores.loc[idx])
        rows.append({
            'Method':           name,
            'ROC-AUC (4-5)':    round(m['roc_auc'], 4),
            'PR-AUC (4-5)':     round(m['pr_auc'],  4),
            'Recall@100 (4-5)': round(topk_recall(y_true.loc[idx], scores.loc[idx]), 4),
            'Spearman (1-5)':   round(labels_raw.loc[idx].corr(
                                    scores.loc[idx], method='spearman'), 4),
        })
    df = pd.DataFrame(rows).set_index('Method')
    display(df.sort_values('ROC-AUC (4-5)', ascending=False))
    return df


def discovery_plot(methods_dict, y_true, highlight=None):
    """Standard cumulative discovery curve."""
    colors     = ['#1f77b4','#d62728','#2ca02c','#ff7f0e','#8c564b',
                  '#e377c2','#17becf','#9467bd','#bcbd22','#e74c3c']
    linestyles = ['-','--','-.', ':','-','-.','--',':','-.','--']

    fig, ax = plt.subplots(figsize=(8, 5))
    for (name, scores), c, ls in zip(methods_dict.items(), colors, linestyles):
        idx = scores.index.intersection(y_true.index)
        lw  = 2.4 if (highlight and name in highlight) else 1.3
        cum = cumulative_sum(scores.loc[idx], y_true.loc[idx])
        ax.plot(cum, label=name, color=c, linestyle=ls, linewidth=lw)
    ax.set_xlabel('Rank (ordered by anomaly score)')
    ax.set_ylabel('Anomalies found (score ≥ 4)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


## 4. Load Data and Protege Baseline

In [ ]:
X_byol = load_features()
cat    = load_catalogue()
X_byol = X_byol.loc[cat.objid]

labels        = cat.set_index('objid')['evaluation_subset_author_ML_score'].loc[X_byol.index]
y_interesting = (labels >= 4).astype(int)
pca_scores    = cat.set_index('objid')['protege_score'].loc[X_byol.index]
pca_scores.name = 'score'

print(f'Dataset   : {X_byol.shape[0]} objects')
print(f'Anomalies : {y_interesting.sum()} / {len(y_interesting)} (score ≥ 4)')
print(f'Protege ROC-AUC (reference): '
      f'{roc_auc_score(y_interesting, pca_scores):.4f}')


## 5. Discover Images

Image filenames are the **Protege rank** of each object (from the README).
`0.png` = the object Protege ranked #0 (most anomalous), `1.png` = rank #1, etc.
The catalogue `protege_rank` column holds these integers.


In [ ]:
IMAGE_DIRS = [
    BASE_DIR / 'data' / 'images' / 'all_images_compressed_part1',
    BASE_DIR / 'data' / 'images' / 'all_images_compressed_part2',
]

# integer protege_rank → file path
image_map_int = {}
for d in IMAGE_DIRS:
    if not d.exists():
        print(f'WARNING: {d} not found')
        continue
    for p in d.glob('*.png'):
        image_map_int[int(p.stem)] = p

# objid → file path via protege_rank column
image_map = {}
for _, row in cat.iterrows():
    rank  = int(row['protege_rank'])
    objid = row['objid']
    if rank in image_map_int:
        image_map[objid] = image_map_int[rank]

objids_found   = [oid for oid in X_byol.index if oid in image_map]
objids_missing = [oid for oid in X_byol.index if oid not in image_map]

print(f'Images found   : {len(image_map_int)}')
print(f'Mapped to objid: {len(image_map)}')
print(f'Missing        : {len(objids_missing)}')


## 6. PyTorch Dataset

A minimal `Dataset` that loads each `.png`, converts to RGB, resizes to 224×224,
and applies ImageNet normalisation — required for pretrained backbone weights.


In [ ]:
class MGCLSDataset(Dataset):
    """
    Loads MGCLS radio images for PatchCore feature extraction.
    Returns (tensor, objid) per item.
    """
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

    def __init__(self, objids, image_map, img_size=224):
        self.objids    = objids
        self.image_map = image_map
        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            # Radio images are single-channel — replicate to 3 channels
            T.Lambda(lambda x: x.repeat(3, 1, 1) if x.shape[0] == 1 else x),
            T.Normalize(mean=self.IMAGENET_MEAN, std=self.IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.objids)

    def __getitem__(self, idx):
        oid  = self.objids[idx]
        path = self.image_map[oid]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), oid
        except Exception as e:
            print(f'  WARNING: failed to load {path}: {e}')
            return torch.zeros(3, 224, 224), oid


dataset    = MGCLSDataset(objids_found, image_map, img_size=224)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
print(f'Dataset : {len(dataset)} images')
print(f'Batches : {len(dataloader)}')


## 7. anomalib Feature Extractor

`TimmFeatureExtractor` from anomalib wraps any timm backbone and returns
feature maps at the requested layers — **without pooling**, preserving
the spatial grid of patch features.

### Layer choice for WideResNet50

| Layer | Output shape | Spatial grid | What it captures |
|---|---|---|---|
| `layer2` | (N, 512, 28, 28) | 28×28 = 784 patches | Local texture, edges, structure |
| `layer3` | (N, 1024, 14, 14) | 14×14 = 196 patches | Higher-level semantics |

Each patch in the spatial grid becomes **one entry** in the memory bank —
giving 784 + 196 = 980 entries per image instead of just 1.
This is what makes anomalib PatchCore more powerful than image-level pooling.


In [ ]:
class TimmFeatureExtractor(nn.Module):
    """
    Multi-layer feature extractor using any timm backbone.
    Replicates the anomalib TimmFeatureExtractor without the anomalib install.

    Returns a dict {layer_name: feature_map_tensor} for each requested layer.
    Feature maps are NOT pooled — full spatial maps are returned so we can
    extract patch-level features.

    Parameters
    ----------
    backbone : str   timm model name, e.g. 'wide_resnet50_2'
    layers   : list  layer names to hook, e.g. ['layer2', 'layer3']
    """

    def __init__(self, backbone: str, layers: list, pre_trained: bool = True):
        super().__init__()
        self.layers  = layers
        self._feats  = {}

        # Load backbone from timm
        self.model = timm.create_model(
            backbone,
            pretrained=pre_trained,
            features_only=False,   # we register hooks manually
        )
        self.model.eval()

        # Register a forward hook on each requested layer
        for name, module in self.model.named_modules():
            if name in layers:
                module.register_forward_hook(self._make_hook(name))

    def _make_hook(self, name):
        def hook(module, input, output):
            self._feats[name] = output
        return hook

    def forward(self, x):
        self._feats = {}
        with torch.no_grad():
            self.model(x)
        return {k: self._feats[k] for k in self.layers}


# ── Verify ────────────────────────────────────────────────────────────────────
BACKBONE = 'wide_resnet50_2'
LAYERS   = ['layer2', 'layer3']

feature_extractor = TimmFeatureExtractor(
    backbone=BACKBONE,
    layers=LAYERS,
    pre_trained=True,
).to(device)
feature_extractor.eval()

for p in feature_extractor.parameters():
    p.requires_grad = False

print(f'Backbone : {BACKBONE}')
print(f'Layers   : {LAYERS}')
print(f'Device   : {device}')
print(f'Frozen   : True')

# Verify output shapes
with torch.no_grad():
    dummy = torch.zeros(2, 3, 224, 224).to(device)
    out   = feature_extractor(dummy)
    for layer, feat in out.items():
        n_patches = feat.shape[2] * feat.shape[3]
        print(f'  {layer}: {tuple(feat.shape)}'
              f'  → {n_patches} patches × {feat.shape[1]} dims')


## 8. Extract Patch Features — Build Memory Bank

For each image we extract the spatial feature maps from both layers,
**adaptive-average-pool** to a fixed patch grid (28×28 for layer2, 14×14 for layer3
→ both resized to a common 28×28), then flatten to individual patch vectors.

All patch vectors from all images are collected into the **memory bank**.


In [ ]:
from sklearn.preprocessing import normalize as sk_normalize

PATCH_SIZE = 28    # all layers resized to this spatial grid
SAVE_PATH  = BASE_DIR / 'data' / 'anomalib_patch_features.npz'

if SAVE_PATH.exists():
    print(f'Loading cached patch features from {SAVE_PATH}')
    cached          = np.load(SAVE_PATH, allow_pickle=True)
    patch_features  = cached['features']   # (N_patches_total, D)
    patch_objids    = list(cached['objids'])    # objid per patch
    print(f'Loaded: {patch_features.shape[0]:,} patches × {patch_features.shape[1]} dims')

else:
    print(f'Extracting patch features for {len(dataset)} images...')
    pool = torch.nn.AdaptiveAvgPool2d((PATCH_SIZE, PATCH_SIZE))

    all_patches = []   # list of (n_patches, D) arrays
    all_objids  = []   # objid repeated for each patch

    for batch_imgs, batch_ids in dataloader:
        batch_imgs = batch_imgs.to(device)

        with torch.no_grad():
            feat_maps = feature_extractor(batch_imgs)

        # Process each layer: pool to common grid, flatten patches
        layer_patches = []
        for layer_name in LAYERS:
            fm    = feat_maps[layer_name]           # (B, C, H, W)
            fm_p  = pool(fm)                        # (B, C, 28, 28)
            # Reshape: (B, C, 28, 28) → (B, 28*28, C) = (B, 784, C)
            B, C, H, W = fm_p.shape
            patches = fm_p.permute(0, 2, 3, 1).reshape(B, H * W, C)
            layer_patches.append(patches.cpu().numpy())

        # Concatenate layers along feature dim: (B, 784, C2+C3)
        combined = np.concatenate(layer_patches, axis=2)  # (B, 784, 1536)

        for i, oid in enumerate(batch_ids):
            img_patches = combined[i]                # (784, 1536)
            # L2-normalise each patch vector
            img_patches = sk_normalize(img_patches, norm='l2')
            all_patches.append(img_patches)
            all_objids.extend([oid] * len(img_patches))

        n_done = len(all_patches) * img_patches.shape[0]
        print(f'  {len(all_patches):4d}/{len(dataset)} images', end='\r')

    print()
    patch_features = np.vstack(all_patches)    # (N_total_patches, D)
    patch_objids   = all_objids

    np.savez_compressed(
        SAVE_PATH,
        features=patch_features,
        objids=np.array(patch_objids)
    )
    print(f'Saved to {SAVE_PATH}')

print(f'\nTotal patches : {patch_features.shape[0]:,}')
print(f'Feature dims  : {patch_features.shape[1]}')
print(f'Patches/image : {patch_features.shape[0] // len(objids_found)}')


## 9. Coreset Subsampling — Memory Bank

With 784 patches per image × 6,161 images = ~4.8 million patch vectors,
a nearest-neighbour search over the full set would be very slow.

**Greedy coreset subsampling** reduces this to a representative subset
by iteratively selecting the patch furthest from all already-selected patches.
This keeps the memory bank maximally spread out in feature space.

`subsample_ratio=0.1` keeps 10% — ~480,000 patches. Adjust based on your RAM.


In [ ]:
def greedy_coreset(features, subsample_ratio=0.1, random_state=42,
                   chunk=2000):
    """
    Greedy furthest-point coreset subsampling.
    chunk: process distances in blocks to limit RAM usage.
    """
    N        = len(features)
    n_select = max(1, int(N * subsample_ratio))
    rng      = np.random.default_rng(random_state)

    selected = [int(rng.integers(N))]
    min_dist = np.full(N, np.inf)

    for step in range(1, n_select):
        last = features[selected[-1]:selected[-1]+1]   # (1, D)
        # Compute distances in chunks to limit memory
        for start in range(0, N, chunk):
            block = features[start:start+chunk]
            dists = np.linalg.norm(block - last, axis=1)
            min_dist[start:start+chunk] = np.minimum(
                min_dist[start:start+chunk], dists)
        selected.append(int(np.argmax(min_dist)))
        if (step + 1) % 5000 == 0 or step == n_select - 1:
            print(f'  Coreset: {step+1:,}/{n_select:,}', end='\r')

    print()
    return np.array(selected)


SUBSAMPLE = 0.1   # reduce if RAM is limited; increase for better accuracy

print(f'Building coreset (subsample_ratio={SUBSAMPLE})...')
print(f'Input : {patch_features.shape[0]:,} patches')

coreset_idx     = greedy_coreset(patch_features, subsample_ratio=SUBSAMPLE)
memory_bank     = patch_features[coreset_idx]

print(f'Memory bank: {memory_bank.shape[0]:,} patches '
      f'({memory_bank.shape[0]/patch_features.shape[0]:.1%} of total)')
print(f'Feature dim: {memory_bank.shape[1]}')


## 10. Image-Level Anomaly Scoring

For each image, compute the maximum patch-level nearest-neighbour distance
across all its patches. This is the PatchCore anomaly score:

```
score(image) = max over patches p in image:
                   min over memory vectors m:
                       distance(p, m)
```

A single unusual patch anywhere in the image drives the image score high.
This is what makes PatchCore sensitive to **local** anomalies — a compact
jet or an unusual morphological feature in one corner of the image gets detected.

We use k-NN (k=3) for robustness: average over the 3 nearest memory patches
rather than just 1.


In [ ]:
from sklearn.metrics import pairwise_distances_chunked

def image_anomaly_scores(patch_features, patch_objids, memory_bank,
                         objids_eval, k=3):
    """
    Compute image-level anomaly scores from patch-level nearest-neighbour distances.

    For each image: score = max patch distance to memory bank.

    Parameters
    ----------
    patch_features : np.ndarray  (N_patches, D)  all patch vectors
    patch_objids   : list        objid for each patch row
    memory_bank    : np.ndarray  (M, D)  coreset subset
    objids_eval    : list        objids to score
    k              : int         neighbours to average

    Returns
    -------
    pd.Series  image-level anomaly scores indexed by objid
    """
    objid_arr = np.array(patch_objids)
    scores    = {}

    N_total  = len(patch_features)
    N_done   = 0
    # Process all patches in chunks; accumulate max per image
    img_max  = {}   # objid → running max patch distance

    for chunk_dists in pairwise_distances_chunked(
            patch_features, memory_bank,
            metric='euclidean', n_jobs=-1, working_memory=512):

        n_chunk  = chunk_dists.shape[0]
        k_actual = min(k, chunk_dists.shape[1])
        topk     = np.partition(chunk_dists, k_actual-1, axis=1)[:, :k_actual]
        patch_scores = topk.mean(axis=1)   # (n_chunk,)

        # Map each patch score to its image
        chunk_ids = objid_arr[N_done : N_done + n_chunk]
        for oid, ps in zip(chunk_ids, patch_scores):
            if oid in img_max:
                img_max[oid] = max(img_max[oid], ps)
            else:
                img_max[oid] = ps

        N_done += n_chunk
        print(f'  Scoring: {N_done:,}/{N_total:,} patches', end='\r')

    print()

    # Restrict to evaluation set
    result = {oid: img_max.get(oid, np.nan) for oid in objids_eval}
    return pd.Series(result, name='score')


print('Computing image-level anomaly scores (k=3)...')
anomalib_scores = image_anomaly_scores(
    patch_features, patch_objids, memory_bank,
    objids_eval=objids_found, k=3
)
anomalib_scores.index = anomalib_scores.index.astype(type(X_byol.index[0]))

print(f'Score range: [{anomalib_scores.min():.4f}, {anomalib_scores.max():.4f}]')


## 11. k-NN Sweep — Find Best k

Test k ∈ {1, 3, 5, 10} to find the most robust neighbourhood size.


In [ ]:
k_scores = {}
print('k sweep:')
for k in [1, 3, 5, 10]:
    s = image_anomaly_scores(
        patch_features, patch_objids, memory_bank,
        objids_eval=objids_found, k=k
    )
    s.index = s.index.astype(type(X_byol.index[0]))
    k_scores[k] = s
    idx = s.index.intersection(y_interesting.index)
    m   = compute_metrics(y_interesting.loc[idx], s.loc[idx])
    print(f'  k={k:2d}  ROC-AUC={m["roc_auc"]:.4f}  '
          f'PR-AUC={m["pr_auc"]:.4f}  '
          f'Recall@100={topk_recall(y_interesting.loc[idx], s.loc[idx]):.4f}')

best_k = max(k_scores, key=lambda k: compute_metrics(
    y_interesting.loc[k_scores[k].index.intersection(y_interesting.index)],
    k_scores[k].loc[k_scores[k].index.intersection(y_interesting.index)]
)['roc_auc'])
print(f'\nBest k = {best_k}')
anomalib_scores_best = k_scores[best_k]


## 12. Backbone Comparison

Try a second backbone (ResNet18 — faster, fewer parameters) and compare.
This tests whether the wider architecture of WideResNet50 is necessary
or if a lighter model achieves similar results.


In [ ]:
def run_backbone(backbone_name, layers, objids_found, image_map,
                 subsample_ratio=0.1):
    """
    Full PatchCore pipeline for a given backbone.
    Returns image-level anomaly scores for objids_found.
    """
    print(f'\n=== Backbone: {backbone_name} ===')

    extractor = TimmFeatureExtractor(
        backbone=backbone_name, layers=layers, pre_trained=True
    ).to(device)
    extractor.eval()
    for p in extractor.parameters():
        p.requires_grad = False

    # Check output shapes
    with torch.no_grad():
        dummy = torch.zeros(2, 3, 224, 224).to(device)
        out   = extractor(dummy)
        for lyr, feat in out.items():
            n_patches = feat.shape[2] * feat.shape[3]
            print(f'  {lyr}: {tuple(feat.shape)} → {n_patches} patches × {feat.shape[1]} dims')

    pool      = torch.nn.AdaptiveAvgPool2d((PATCH_SIZE, PATCH_SIZE))
    ds        = MGCLSDataset(objids_found, image_map, img_size=224)
    dl        = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)

    all_p  = []
    all_o  = []
    for batch_imgs, batch_ids in dl:
        batch_imgs = batch_imgs.to(device)
        with torch.no_grad():
            feat_maps = extractor(batch_imgs)
        layer_patches = []
        for lyr in layers:
            fm   = pool(feat_maps[lyr])
            B, C, H, W = fm.shape
            pts  = fm.permute(0,2,3,1).reshape(B, H*W, C).cpu().numpy()
            layer_patches.append(pts)
        combined = np.concatenate(layer_patches, axis=2)
        for i, oid in enumerate(batch_ids):
            pts_norm = sk_normalize(combined[i], norm='l2')
            all_p.append(pts_norm)
            all_o.extend([oid] * len(pts_norm))
        print(f'  {len(all_p):4d}/{len(ds)} images', end='\r')

    print()
    feats = np.vstack(all_p)
    cidx  = greedy_coreset(feats, subsample_ratio=subsample_ratio)
    bank  = feats[cidx]
    s     = image_anomaly_scores(feats, all_o, bank, objids_found, k=best_k)
    s.index = s.index.astype(type(X_byol.index[0]))
    idx   = s.index.intersection(y_interesting.index)
    m     = compute_metrics(y_interesting.loc[idx], s.loc[idx])
    print(f'  ROC-AUC={m["roc_auc"]:.4f}  '
          f'PR-AUC={m["pr_auc"]:.4f}  '
          f'Recall@100={topk_recall(y_interesting.loc[idx], s.loc[idx]):.4f}')
    return s


# ResNet18 uses different layer names
resnet18_scores = run_backbone(
    'resnet18',
    layers=['layer2', 'layer3'],
    objids_found=objids_found,
    image_map=image_map,
    subsample_ratio=0.1,
)


## 13. Evaluation — All anomalib PatchCore Variants

In [ ]:
# Restrict Protege to common eval ids
eval_idx = anomalib_scores_best.index.intersection(y_interesting.index)

methods_anomalib = {
    'PCA (Protege)':                    pca_scores.loc[eval_idx],
    f'anomalib WideResNet50 (k={best_k})': anomalib_scores_best.loc[eval_idx],
    'anomalib ResNet18':                resnet18_scores.loc[
                                            resnet18_scores.index.intersection(eval_idx)],
}

df_eval = show_results(methods_anomalib, y_interesting, labels)


In [ ]:
discovery_plot(
    methods_anomalib,
    y_interesting,
    highlight=['PCA (Protege)',
               f'anomalib WideResNet50 (k={best_k})']
)


## 14. Full Comparison — All Methods Across All Notebooks

Combine every score from the BYOL-based evaluation notebook and both
PatchCore notebooks into one final ranked table and discovery curve.

Import the scores from previous work by loading `patchcore_features.parquet`
if it exists (from the custom PatchCore notebook).


In [ ]:
# Load custom PatchCore scores if available
CUSTOM_PC_PATH = BASE_DIR / 'data' / 'patchcore_features.parquet'

all_methods_final = {
    'PCA (Protege)': pca_scores,
}

# Add anomalib scores
all_methods_final[f'anomalib WideResNet50 (k={best_k})'] = anomalib_scores_best
all_methods_final['anomalib ResNet18'] = resnet18_scores

# Add custom PatchCore scores if available
if CUSTOM_PC_PATH.exists():
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import pairwise_distances_chunked as pdc

    df_custom = pd.read_parquet(CUSTOM_PC_PATH)
    # Re-score with k-NN quickly (image-level, not patch-level)
    common = [oid for oid in X_byol.index if oid in df_custom.index]
    X_c    = StandardScaler().fit_transform(df_custom.loc[common].values)
    raw    = []
    for chunk in pdc(X_c, X_c, metric='euclidean',
                     n_jobs=-1, working_memory=256):
        topk = np.partition(chunk, 3, axis=1)[:, :3]
        raw.append(topk.mean(axis=1))
    custom_scores = pd.Series(np.concatenate(raw), index=common, name='score')
    all_methods_final['Custom PatchCore (image-level)'] = custom_scores
    print('Custom PatchCore scores added.')
else:
    print('Custom PatchCore parquet not found — skipping.')

df_final = show_results(all_methods_final, y_interesting, labels)


In [ ]:
# Top-3 excluding Protege
all_aucs = {}
for name, scores in all_methods_final.items():
    idx = scores.index.intersection(y_interesting.index)
    all_aucs[name] = roc_auc_score(y_interesting.loc[idx], scores.loc[idx])

ranked = sorted(
    {k: v for k, v in all_aucs.items() if k != 'PCA (Protege)'}.items(),
    key=lambda x: x[1], reverse=True
)
top3_names = [n for n, _ in ranked[:3]]
print('Top-3 methods (excl. Protege):')
for n, auc in ranked[:3]:
    print(f'  {n}: {auc:.4f}')

final_plot = {'PCA (Protege)': pca_scores}
for n in top3_names:
    final_plot[n] = all_methods_final[n]

discovery_plot(
    final_plot,
    y_interesting,
    highlight=['PCA (Protege)'] + top3_names
)


## 15. How to Read These Results

### The progression from custom PatchCore to anomalib

| Aspect | Custom (previous notebook) | anomalib (this notebook) |
|---|---|---|
| Aggregation | 1 vector per image (global pool) | 784 patch vectors per image |
| Memory bank | ~6K image vectors | ~4.8M patch vectors → subsampled |
| Score | Mean k-NN distance | **Max** patch-level k-NN distance |
| Backbone | WideResNet50 only | WideResNet50 + ResNet18 |

The **max** vs **mean** difference is important. Image-level pooling averages
away local anomalies — a single unusual patch is diluted by 783 normal patches.
Using the maximum patch score means one anomalous region anywhere in the image
is enough to flag it.

### If anomalib beats custom PatchCore
Patch-level aggregation is adding genuine signal — local radio morphology features
that survive in the spatial maps but are lost after global pooling.

### If anomalib beats or matches Protege
This is the breakthrough result: a fully unsupervised method with no human labels
matches a system built with iterative human feedback. Report the exact metrics.

### If anomalib is still below Protege
The gap quantifies the value of human labels. The remaining next step would be
to combine anomalib's PatchCore scores into the ensemble from the main notebook —
an ensemble of the best BYOL-based method + PatchCore often outperforms either alone.
